License

- Content (explanatory text, figures): CC BY 4.0 — see /LICENSE-CONTENT
- Code cells and standalone code files: MIT License — see /LICENSE-CODE

Attribution example: Getman, R. (2026). CBE 3610 Python Notebooks and Course Materials. https://github.com/dr-rachel-bg-teaching/CBE3610


# E08 Multiple Reactions in an isothermal CSTR

In [ ]:
import numpy as np
from scipy.optimize import fsolve

## Given information

In [ ]:
v0 = 100 # L/min
V = 400 # L
C0 = np.array([3, 0, 0, 0, 0])  # [CA0, CB0, CC0, CD0, CE0], mol/L
F0 = v0 * C0

rate_constants = (7, 3, 2)  # (k1A, k2D, k3E). units: min^-1, L^2/mol^2/min, L/mol/min
nu = np.array([[-1, 1/3, 1/3, 0, 0],
               [-1/3, 0, -2/3, 1, 0],
               [0, 0, -1, -4/3, 1]])

## CSTR mole balances

In [ ]:
def CSTR(F, nu, F0, rates_args, v0, V):
  C = F / v0

  # Reaction rate laws
  r = np.array([
    rate_constants[0] * C[0],              # r1 = k1A * CA
    rate_constants[1] * (C[2]**2) * C[0],  # r2 = k2D * CC^2 * CA
    rate_constants[2] * C[3] * C[2]        # r3 = k3E * CD * CC
  ])

  # Mole balance: F0 - F + V * (nu^T . r)
  return F0 - F + V * np.dot(nu.T, r)

## Solve

In [ ]:
F_guess = np.array([0.1, 0.93, 0.51, 0.049, 0.02]) * v0
F_sol = fsolve(CSTR, F_guess, args=(nu, F0, rate_constants, v0, V))

print("Final Concentrations (mol/L):", F_sol / v0)
print("Guess minus Actual Flowrates: ", F_guess - F_sol)

# Re-run balance checks instantly with the solved values
final_balances = CSTR(F_sol, nu, F0, rate_constants, v0, V)
print("Mole Balance Residuals (Should be ~0):", np.round(final_balances, 6))

Final Concentrations (mol/L): [0.09978161 0.93129502 0.51615417 0.04903434 0.20247422]
Guess minus Actual Flowrates:  [ 2.18390335e-02 -1.29502354e-01 -6.15416709e-01 -3.43365199e-03
 -1.82474217e+01]
Mole Balance Residuals (Should be ~0): [-0.  0. -0. -0.  0.]
